# DL 01 — Data preparation and validation

**Project 9 — Indoor location prediction using Wi-Fi signals (Signal2Space), Phase 2 (deep learning).**

This notebook prepares the inputs for every DL notebook. It deliberately **reuses the AML feature contract** written by
`Machine Learning Pipeline/ML_01_data_preparation_features.ipynb`, so AML and DL are trained and scored on identical rows:

| Block | Columns | Source of fitted values |
|---|---|---|
| Scaled RSSI | 520 `RSSI_Z__WAPxxx` | per-WAP mean/std fitted by ML_01 on **internal training only**; undetected (`100`) imputed one unit below each WAP's lowest standardized training reading |
| Missingness indicators | 520 `SEEN__WAPxxx` | 1 if the access point was detected, else 0 |
| Signal summaries | 6 (`n_heard`, `max_rssi_dbm`, `mean_rssi_dbm`, `std_rssi_dbm`, `n_strong`, `top1_ap_index`) | raw in the AML table; **standardized here** with internal-training mean/std because a neural network needs comparable input scales |

It also fits, on internal training only, the coordinate standardizer (the network regresses standardized `LONGITUDE`/`LATITUDE`) and the building/floor label mappings.

**Held-out discipline.** The supplied `validationData.csv` features are only schema-checked and *described* here (overlap and identifier counts). Nothing is fitted on them.

In [1]:
from pathlib import Path
import sys

def find_project_root(start=None):
    candidate = Path.cwd() if start is None else Path(start)
    for parent in (candidate, *candidate.parents):
        if (parent / "data/raw").is_dir() and (parent / "src/signal2space_dl").is_dir():
            return parent
    raise FileNotFoundError("Run this notebook from inside the Signal2Space repository")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import platform
import numpy as np
import pandas as pd
from IPython.display import Image, display
from signal2space_dl import config as C, data

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
print(f"Python {platform.python_version()} | pandas {pd.__version__}")
print("Repository root:", PROJECT_ROOT)

Python 3.13.18 | pandas 3.1.1
Repository root: /home/user/Signal2Space


## 1. Validate the AML feature contract and fit the train-only DL transforms

`data.prepare_and_report` checks columns, partitions, finite values, binary indicators, disjoint grouped splits, that the AML RSSI scaler is centred on internal-training observations, and rebuilds ML_01's `GroupShuffleSplit` from the raw CSV to confirm identical rows.

In [2]:
result = data.prepare_and_report()
for line in result["checks"]:
    print("✓", line)

✓ internal_train: all 1046 feature columns and targets/metadata present
✓ internal_train: 16085 rows, unique ROW_ID, SOURCE_PARTITION=trainingData
✓ internal_train: finite predictors, binary indicators, complete targets
✓ internal_validation: all 1046 feature columns and targets/metadata present
✓ internal_validation: 3852 rows, unique ROW_ID, SOURCE_PARTITION=trainingData
✓ internal_validation: finite predictors, binary indicators, complete targets
✓ internal splits: disjoint ROW_IDs and disjoint (BUILDINGID, FLOOR, SPACEID) groups
✓ AML RSSI scaler verified as fitted on internal-training observations (per-WAP mean ~ 0)
✓ raw rebuild: GroupShuffleSplit(test_size=0.20, random_state=42) reproduces 16085 / 3852 rows and targets match trainingData.csv
✓ heldout_validation: all 1046 feature columns and targets/metadata present
✓ heldout_validation: 1111 rows, unique ROW_ID, SOURCE_PARTITION=validationData
✓ heldout_validation: finite predictors, binary indicators, complete targets
✓ heldou

## 2. Fitted preprocessing values

Saved to `models/dl/preprocessor.json` and `data/tables/dl_01_preprocessing_summary.csv`.

In [3]:
display(result["summary"])

,item,value
0,fit_split,internal_train
1,fit_rows,16085
2,n_features,1046
3,rssi_features,520
4,seen_indicators,520
5,summary_features,6
6,building_classes,"[0, 1, 2]"
7,floor_classes,"[0, 1, 2, 3, 4]"
8,coord_mean_longitude,-7463.031087
9,coord_mean_latitude,4864869.910686


## 3. Partition overlap (descriptive only)

Counts below compare each evaluation split with the internal-training rows. They describe how hard each split is; they are never used for fitting or selection.

* *Identical fingerprint* = same 1,040 RSSI/indicator values.
* *Exact position* = same (`LONGITUDE`, `LATITUDE`) pair.

In [4]:
display(result["overlap"])

,comparison,rows,rows_with_identical_fingerprint_in_internal_train,duplicate_fingerprints_within_split,rows_with_exact_position_in_internal_train,distinct_positions,zero_detected_ap_rows
0,internal_train vs internal_validation,3852,156,156,1428,174,5
1,internal_train vs heldout_validation,1111,0,5,26,1044,0


## 4. Identifier and collection-time summary

`USERID` and `PHONEID` are retained for post-hoc robustness analysis in DL_05 but are **not** model inputs. The held-out file was collected months after the training file, its `USERID` is a placeholder (`0`), and most of its phones do not appear in training, so it measures temporal and device generalisation as well as location accuracy.

In [5]:
display(result["identity"])

,split,rows,distinct_USERID,distinct_PHONEID,USERID_values,PHONEID_values,rows_with_PHONEID_seen_in_internal_train,timestamp_min_utc,timestamp_max_utc
0,internal_train,16085,18,16,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18","1,3,6,7,8,10,11,13,14,16,17,18,19,22,23,24",16085,2013-05-30 10:15:24,2013-06-20 14:15:09
1,internal_validation,3852,18,16,"1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18","1,3,6,7,8,10,11,13,14,16,17,18,19,22,23,24",3852,2013-05-30 10:16:31,2013-06-20 14:15:45
2,heldout_validation,1111,1,11,0,"0,2,4,5,9,12,13,14,15,20,21",397,2013-09-19 08:12:47,2013-10-08 15:57:16


## Conclusion

The AML split and feature contract are reused unchanged and verified. DL-specific transforms (summary standardization, coordinate standardization, label mappings) are fitted on internal training only and saved. Next: DL_02–DL_04 train single-task ablation networks; DL_05 trains and evaluates the shared-trunk multitask model.